In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import login
login("APYKEY")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, re, pandas as pd

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

print("Loaded successfully")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "BioMistral/BioMistral-7B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
print("BioMistral loaded!")

In [ ]:
import pandas as pd

df = pd.read_excel("Dataset")
df.rename(columns={'label': 'Hopkins_category'}, inplace=True)

In [ ]:
import re

results = []

for idx, row in prompts_df.iterrows():

    # ✅ print progress every 100 rows
    if idx % 100 == 0:
        print(f"Processing row {idx}")

    prompt_text = row["prompt"]
    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=True,
            temperature=0.7,
            top_p=0.9
        )

    generated_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    # Extract YES / NO
    matches = re.findall(r"\b(YES|NO)\b", generated_text.upper())
    prediction = matches[-1] if matches else "UNKNOWN"

    results.append({
        "ID": row["ID"],
        "Hopkins_category": row["Hopkins_category"],
        "prediction": prediction
    })

predictions_df = pd.DataFrame(results)

In [ ]:
df_clean = predictions_df.dropna(subset=["Hopkins_category", "prediction"])

y_true = df_clean["Hopkins_category"].astype(str)
y_pred = df_clean["prediction"].astype(str)
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred))